# 04 — XAI Evaluation — ResNet-50 (CRC Histology)

XAI evaluation of **ResNet-50** (CNN baseline) on colorectal cancer histology tiles.
- **Test set**: CRC-VAL-HE-7K (external, 7 180 images, 9 classes)
- **XAI methods**: Grad-CAM · Integrated Gradients · LRP

**Transform note (consistency check)**  
Training used `val_transform = Resize(224) → Normalize(ImageNet) → ToTensorV2`.  
This notebook applies the same pipeline via `build_crc_transforms(split='test')`, which is identical.

**Environment**: Kaggle (GPU T4 / P100)  
**Phase 1 only** — CRC histology / ResNet-50 CNN baseline.

## 0. Kaggle Setup — Clone repo & install deps

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical

In [ ]:
!pip install --upgrade pip
!pip install -q timm==0.9.12 albumentations loguru
!pip install -q captum==0.7.0
!pip install grad-cam
!pip install -q PyDrive2
!pip install -q scikit-learn opencv-python-headless

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## 1. Configuration

In [ ]:
import os
import sys
import csv
import json
from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torch.utils.data import DataLoader, Subset

PROJECT_ROOT = "/kaggle/working/xai-vit-medical"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES, build_crc_transforms
from src.utils.seed import set_seed

# ---- Paths ----
TRAINVAL_ROOT = Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K")
TEST_ROOT     = Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K")

# path to model points
CHECKPOINT_PATH= "/kaggle/input/models/youssefnouiouar1/crt-resnet50/pytorch/default/1/resnet50_best.pth"


SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/xai/resnet50")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Subset ----
N_IMAGES_PER_CLASS = 50   # images per class from CRC-VAL-HE-7K
BATCH_SIZE   = 16
NUM_WORKERS  = 2
IMAGE_SIZE   = 224
SEED         = 42
CLASS_NAMES  = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES  = len(CLASS_NAMES)

# ---- Faithfulness metrics ----
FAITH_N_STEPS        = 50
INSERTION_BASELINE   = "black"   # black | blurred | mean
DELETION_REPLACEMENT = "black"

# ---- XAI methods ----
# Attention Rollout / Generic Attention are NOT applicable to CNN → excluded
METHODS_TO_RUN = ["gradcam", "integrated_gradients", "lrp"]

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Classes: {CLASS_NAMES}")
print(f"Output : {SAVE_DIR}")

## 2. Model — ResNet-50

We load the best checkpoint saved by `01_train_cnn.ipynb`.

In [ ]:
def build_resnet50(num_classes: int, pretrained: bool = False) -> nn.Module:
    return timm.create_model("resnet50", pretrained=pretrained, num_classes=num_classes)


def load_checkpoint(model: nn.Module, ckpt_path: Path) -> tuple[nn.Module, dict]:
    state = torch.load(ckpt_path, map_location="cpu")
    # Checkpoint format from 01_train_cnn.ipynb: {'model_state_dict': ..., 'epoch': ..., ...}
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        meta = {k: v for k, v in state.items() if k != "model_state_dict"}
    elif "state_dict" in state:
        cleaned = {k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}
        model.load_state_dict(cleaned)
        meta = {}
    else:
        cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
        model.load_state_dict(cleaned)
        meta = {}
    return model, meta


model = build_resnet50(num_classes=NUM_CLASSES)
model, ckpt_meta = load_checkpoint(model, CHECKPOINT_PATH)
model = model.to(DEVICE).eval()

print(f"Checkpoint epoch   : {ckpt_meta.get('epoch', '?')}")
print(f"Checkpoint val_acc : {ckpt_meta.get('val_acc', '?')}")
print(f"Model parameters   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Dataset & Test Subset

**Transform consistency**
- Training notebook `val_transform`: `Resize(224) → Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]) → ToTensorV2`
- `build_crc_transforms(split='test')`: identical pipeline — confirmed consistent.

In [ ]:
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.25,
    random_state=SEED,
)

# return_id=True → DataLoader yields (image, label, image_id) triples
test_dataset = CRCHistologyDataset(
    split="test",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=True,
)

# Select N_IMAGES_PER_CLASS images per class in dataset order (deterministic)
class_counts: dict[int, int] = defaultdict(int)
subset_indices: list[int] = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_IMAGES_PER_CLASS:
        subset_indices.append(idx)
        class_counts[lbl] += 1
    if all(class_counts[c] >= N_IMAGES_PER_CLASS for c in range(NUM_CLASSES)):
        break

# _samples is a list of (Path, int) tuples
samples_meta: list[tuple[Path, int]] = [test_dataset._samples[i] for i in subset_indices]

eval_dataset = Subset(test_dataset, subset_indices)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Images per class:")
for ci, cn in enumerate(CLASS_NAMES):
    print(f"  {cn}: {class_counts[ci]}")
print(f"Total selected: {len(subset_indices)}")
print(f"Batches: {len(eval_loader)}")

## 4. XAI Helper Functions

In [ ]:
from src.xai.classical.gradcam import run_gradcam
from src.xai.classical.integrated_gradients import run_integrated_gradients
from src.xai.classical.lrp import run_lrp


# ---- Minimal config objects (replaces Hydra for standalone use) ----
def _gradcam_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        variant="GradCAM",
        aug_smooth=False,
        eigen_smooth=False,
        reshape_transform=SimpleNamespace(
            enabled=False,   # ResNet-50 is a CNN — no patch reshape needed
            height=14, width=14, num_extra_tokens=1,
        ),
        postprocess=SimpleNamespace(
            relu=True, normalize="minmax",
            upsample_mode="bilinear", upsample_to_input_size=True,
        ),
    )


def _ig_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        n_steps=50,
        internal_batch_size=8,
        method="gausslegendre",
        baseline=SimpleNamespace(type="black_image", blur_sigma=10.0),
        postprocess=SimpleNamespace(
            reduce="sum", abs=True, positive_only=False, normalize="minmax",
        ),
    )


def _lrp_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        gamma=0.25, epsilon=0.25, alpha=2.0, beta=1.0,
        postprocess=SimpleNamespace(
            reduce="sum", abs=True, positive_only=False, normalize="minmax",
        ),
    )


# Target layer for Grad-CAM on ResNet-50: last residual block
GRADCAM_TARGET_LAYER = model.layer4[-1]


def compute_saliency(
    method: str,
    images: torch.Tensor,
    targets: list[int],
) -> torch.Tensor:
    """Dispatch to the appropriate XAI function."""
    if method == "gradcam":
        return run_gradcam(
            model=model,
            images=images,
            targets=targets,
            target_layer=GRADCAM_TARGET_LAYER,
            cfg=_gradcam_cfg(),
        )
    if method == "integrated_gradients":
        return run_integrated_gradients(
            model=model,
            images=images,
            targets=targets,
            cfg=_ig_cfg(),
        )
    if method == "lrp":
        return run_lrp(
            model=model,
            images=images,
            targets=targets,
            cfg=_lrp_cfg(),
        )
    raise ValueError(f"Unknown method: {method}")


def model_predict(images: torch.Tensor) -> list[int]:
    with torch.no_grad():
        logits = model(images)
        if hasattr(logits, "logits"):
            logits = logits.logits
        return torch.argmax(logits, dim=1).tolist()


def overlay_heatmap(
    image_chw: torch.Tensor,
    saliency_hw: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:
    """Blend ImageNet-normalized image with a jet heatmap."""
    img = image_chw.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = np.clip(img * std + mean, 0.0, 1.0)

    sal = saliency_hw.astype(np.float32)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)

    heat = cv2.applyColorMap((sal * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


print("XAI helpers ready")
print(f"Grad-CAM target layer: {GRADCAM_TARGET_LAYER}")

## 5. Faithfulness Metrics (Insertion / Deletion AUC)

In [ ]:
def _apply_mask(
    image: torch.Tensor,
    saliency: torch.Tensor,
    fraction: float,
    mode: str,          # "insertion" | "deletion"
    baseline: str = "black",
) -> torch.Tensor:
    """Reveal or mask the top-fraction pixels by saliency."""
    C, H, W = image.shape
    n_pixels = H * W
    k = max(1, int(fraction * n_pixels))

    flat_sal = saliency.flatten()
    topk_idx = torch.topk(flat_sal, k, largest=True).indices
    mask = torch.zeros(n_pixels, device=image.device)
    mask[topk_idx] = 1.0
    mask = mask.reshape(1, H, W)

    if baseline == "black":
        ref = torch.zeros_like(image)
    elif baseline == "blurred":
        ref = F.avg_pool2d(image.unsqueeze(0), kernel_size=31, stride=1, padding=15).squeeze(0)
    else:  # mean
        ref = image.mean(dim=(-2, -1), keepdim=True).expand_as(image)

    if mode == "insertion":
        return ref * (1 - mask) + image * mask
    else:  # deletion
        return image * (1 - mask) + ref * mask


@torch.no_grad()
def _class_prob(img_bchw: torch.Tensor, class_idx: int) -> float:
    logits = model(img_bchw.to(DEVICE))
    if hasattr(logits, "logits"):
        logits = logits.logits
    return float(torch.softmax(logits, dim=1)[0, class_idx].item())


def insertion_deletion_auc(
    image: torch.Tensor,
    saliency: torch.Tensor,
    target_class: int,
    n_steps: int = FAITH_N_STEPS,
    baseline: str = INSERTION_BASELINE,
) -> dict:
    """Compute insertion and deletion curves + AUC for one image."""
    fractions = np.linspace(0, 1, n_steps + 1)
    ins_probs, del_probs = [], []

    sal_t = saliency.to(image.device)

    for frac in fractions:
        ins_img = _apply_mask(image, sal_t, frac, "insertion", baseline).unsqueeze(0)
        del_img = _apply_mask(image, sal_t, frac, "deletion", baseline).unsqueeze(0)
        ins_probs.append(_class_prob(ins_img, target_class))
        del_probs.append(_class_prob(del_img, target_class))

    ins_auc = float(np.trapezoid(ins_probs, fractions))
    del_auc = float(np.trapezoid(del_probs, fractions))
    return {
        "insertion_auc": ins_auc,
        "deletion_auc": del_auc,
        "faithfulness": ins_auc - del_auc,
        "insertion_curve": np.array(ins_probs, dtype=np.float32),
        "deletion_curve": np.array(del_probs, dtype=np.float32),
    }


print("Faithfulness metric helpers ready")

## 6. Run XAI Methods

In [ ]:
from tqdm.notebook import tqdm


def evaluate_method(method_name: str) -> dict:
    method_dir = SAVE_DIR / method_name
    overlay_dir  = method_dir / "overlays"
    saliency_dir = method_dir / "saliency"
    for d in (method_dir, overlay_dir, saliency_dir):
        d.mkdir(parents=True, exist_ok=True)

    curves_csv = method_dir / "curves.csv"
    auc_csv    = method_dir / "auc_scores.csv"

    with open(curves_csv, "w", newline="") as f:
        csv.writer(f).writerow(["image_id", "method", "step", "type", "probability"])
    with open(auc_csv, "w", newline="") as f:
        csv.writer(f).writerow(
            ["image_id", "label_idx", "label_name", "pred_idx", "pred_name",
             "method", "insertion_auc", "deletion_auc", "faithfulness"]
        )

    ins_aucs, del_aucs = [], []
    ins_curves_list, del_curves_list = [], []
    n_correct = 0
    n_processed = 0

    for images, labels, image_ids in tqdm(eval_loader, desc=method_name):
        images = images.to(DEVICE, non_blocking=True)
        preds = model_predict(images)
        targets = preds  # explain the predicted class (model's view)
        n_correct += sum(p == l.item() for p, l in zip(preds, labels))

        saliency_batch = compute_saliency(method_name, images, targets)
        if not torch.is_tensor(saliency_batch):
            saliency_batch = torch.tensor(saliency_batch)
        saliency_batch = saliency_batch.detach().cpu()

        for i in range(images.shape[0]):
            image_id  = image_ids[i]
            label_idx = int(labels[i].item())
            pred_idx  = int(preds[i])
            sal       = saliency_batch[i].numpy().astype(np.float32)

            faith = insertion_deletion_auc(
                image=images[i].detach().cpu(),
                saliency=torch.from_numpy(sal),
                target_class=pred_idx,
            )

            ins_aucs.append(faith["insertion_auc"])
            del_aucs.append(faith["deletion_auc"])
            ins_curves_list.append(faith["insertion_curve"])
            del_curves_list.append(faith["deletion_curve"])

            with open(curves_csv, "a", newline="") as f:
                w = csv.writer(f)
                for step, p in enumerate(faith["insertion_curve"]):
                    w.writerow([image_id, method_name, step, "insertion", float(p)])
                for step, p in enumerate(faith["deletion_curve"]):
                    w.writerow([image_id, method_name, step, "deletion", float(p)])

            with open(auc_csv, "a", newline="") as f:
                csv.writer(f).writerow([
                    image_id, label_idx, CLASS_NAMES[label_idx],
                    pred_idx, CLASS_NAMES[pred_idx],
                    method_name,
                    faith["insertion_auc"], faith["deletion_auc"], faith["faithfulness"],
                ])

            # Save saliency + overlay
            stem = Path(image_id).stem
            np.save(saliency_dir / f"{stem}_saliency.npy", sal)
            overlay = overlay_heatmap(images[i].detach().cpu(), sal)
            cv2.imwrite(
                str(overlay_dir / f"{stem}_overlay.jpg"),
                cv2.cvtColor((overlay * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
            )

            n_processed += 1

    mean_ins = float(np.mean(ins_aucs))
    mean_del = float(np.mean(del_aucs))
    accuracy = n_correct / n_processed

    mean_ins_curve = np.mean(np.stack(ins_curves_list), axis=0)
    mean_del_curve = np.mean(np.stack(del_curves_list), axis=0)

    summary = {
        "method": method_name,
        "n_images": n_processed,
        "accuracy": accuracy,
        "mean_insertion_auc": mean_ins,
        "mean_deletion_auc": mean_del,
        "mean_faithfulness": mean_ins - mean_del,
    }
    with open(method_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # Plot mean curves
    x = np.linspace(0, 1, len(mean_ins_curve))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(x, mean_ins_curve)
    axes[0].set(title=f"{method_name} — Mean Insertion Curve",
                xlabel="Fraction of pixels inserted", ylabel="Class probability")
    axes[0].grid(True)
    axes[1].plot(x, mean_del_curve, color="tab:red")
    axes[1].set(title=f"{method_name} — Mean Deletion Curve",
                xlabel="Fraction of pixels deleted", ylabel="Class probability")
    axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(str(method_dir / "curves.png"), dpi=150)
    plt.show()

    print(f"[{method_name}] n={n_processed}  acc={accuracy:.3f}")
    print(f"[{method_name}] insertion AUC = {mean_ins:.4f}")
    print(f"[{method_name}] deletion  AUC = {mean_del:.4f}")
    print(f"[{method_name}] faithfulness  = {mean_ins - mean_del:.4f}")

    return summary


print("evaluate_method() defined — ready to run")

In [ ]:
all_summaries: list[dict] = []

for method in METHODS_TO_RUN:
    print("\n" + "=" * 72)
    print(f"Running: {method}")
    print("=" * 72)
    try:
        summary = evaluate_method(method)
        all_summaries.append(summary)
    except Exception as e:
        print(f"[ERROR] {method}: {type(e).__name__}: {e}")
        all_summaries.append({"method": method, "error": str(e)})

print("\n" + "=" * 72)
print("Summary")
print("=" * 72)
for s in all_summaries:
    if "error" in s:
        print(f"  {s['method']:25s} ERROR: {s['error']}")
    else:
        print(
            f"  {s['method']:25s} "
            f"ins={s['mean_insertion_auc']:.4f}  "
            f"del={s['mean_deletion_auc']:.4f}  "
            f"faith={s['mean_faithfulness']:.4f}"
        )

## 7. Visual Inspection — Sample Overlays

In [ ]:
N_DISPLAY = 3  # images per method to display

for s in all_summaries:
    if "error" in s:
        continue
    method = s["method"]
    overlay_dir = SAVE_DIR / method / "overlays"
    candidates = sorted(overlay_dir.glob("*.jpg"))[:N_DISPLAY]
    if not candidates:
        continue

    fig, axes = plt.subplots(1, len(candidates), figsize=(5 * len(candidates), 5))
    if len(candidates) == 1:
        axes = [axes]
    fig.suptitle(f"Method: {method}", fontsize=14)

    for ax, img_path in zip(axes, candidates):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.stem.replace("_overlay", ""), fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 8. Per-Class Saliency Montage

Show one representative overlay per class for each XAI method.

In [ ]:
def show_class_montage(method_name: str, n_per_class: int = 1) -> None:
    """Display one overlay per class, grouped by class."""
    overlay_dir = SAVE_DIR / method_name / "overlays"
    if not overlay_dir.exists():
        print(f"No overlays for {method_name}")
        return

    # Build a class → [image_path] mapping from samples_meta
    class_to_paths: dict[int, list[Path]] = defaultdict(list)
    for img_path, label_idx in samples_meta:
        stem = img_path.stem
        overlay = overlay_dir / f"{stem}_overlay.jpg"
        if overlay.exists():
            class_to_paths[label_idx].append(overlay)

    n_classes = NUM_CLASSES
    fig, axes = plt.subplots(
        n_classes, n_per_class,
        figsize=(4 * n_per_class, 4 * n_classes),
        squeeze=False,
    )
    fig.suptitle(f"ResNet-50 — {method_name} — per-class overlays", fontsize=14)

    for row, class_idx in enumerate(range(n_classes)):
        paths = class_to_paths[class_idx][:n_per_class]
        for col in range(n_per_class):
            ax = axes[row][col]
            if col < len(paths):
                img = cv2.cvtColor(cv2.imread(str(paths[col])), cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                # Show the class name at the top of each image
                ax.set_title(CLASS_NAMES[class_idx], fontsize=11)
            else:
                ax.set_visible(False)
            ax.axis("off")

    plt.tight_layout()
    out = SAVE_DIR / method_name / "class_montage.png"
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")


for s in all_summaries:
    if "error" not in s:
        show_class_montage(s["method"], n_per_class=2)


## 9. Save Results & Upload to Google Drive

In [ ]:
# Global summary
global_summary_path = SAVE_DIR / "summary_all_methods.json"
with open(global_summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)

# Subset manifest (reproducibility)
manifest_path = SAVE_DIR / "subset_manifest.csv"
with open(manifest_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image_path", "label_idx", "label_name"])
    for img_path, label_idx in samples_meta:   # samples_meta is list of (Path, int) tuples
        w.writerow([str(img_path), label_idx, CLASS_NAMES[label_idx]])

print(f"Saved: {global_summary_path}")
print(f"Saved: {manifest_path}")

In [ ]:
RESULTS_DRIVE_FOLDER = "1eq-Jt6O6gO0Ck_oQYwmmc2qrCVLfKlec"  # update if different

files_to_upload = [
    global_summary_path,
    manifest_path,
]
# Also upload per-method summaries and curve plots
for s in all_summaries:
    if "error" in s:
        continue
    method_dir = SAVE_DIR / s["method"]
    files_to_upload += [
        method_dir / "summary.json",
        method_dir / "curves.png",
        method_dir / "class_montage.png",
        method_dir / "auc_scores.csv",
    ]

for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}")
        continue
    drive_file = drive.CreateFile({
        "title": fpath.name,
        "parents": [{"id": RESULTS_DRIVE_FOLDER}],
    })
    drive_file.SetContentFile(str(fpath))
    drive_file.Upload()
    print(f"  Uploaded: {fpath.name}")

print("Done.")

In [ ]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done")